In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


In [67]:
dir = os.getcwd()

# Feature extraction

In [68]:
data = pd.read_excel( f"{dir}/DMT Questionnaires and Subject data.xlsx",
                     sheet_name = "Subject data",
                     skiprows = [3, 6, 9, 14, 17, 24, 32],
                     index_col = False )
data.loc[data['DMT - Previous Experiences'] == 1, ['DMT - Previous Experiences']] = 0
data.loc[data['DMT - Previous Experiences'] > 1, ['DMT - Previous Experiences']] = 1
dmt_exp = data["DMT - Previous Experiences"]

Read precalculated PSD and calculate averge value per band and subject

In [69]:
aal90_labels = pd.read_csv( f"{dir}/PSD/AAL90.csv", delimiter = ";", index_col = 0 )["Label"].to_list()
pre_alpha = pd.read_csv( f"{dir}/PSD/alpha.csv", names = aal90_labels, skiprows=[10] )
pre_beta = pd.read_csv( f"{dir}/PSD/beta.csv", names = aal90_labels, skiprows=[10] )
pre_delta = pd.read_csv( f"{dir}/PSD/delta.csv", names = aal90_labels, skiprows=[10] )
pre_gamma1 = pd.read_csv( f"{dir}/PSD/gamma1.csv", names = aal90_labels, skiprows=[10] )
pre_gamma2 = pd.read_csv( f"{dir}/PSD/gamma2.csv", names = aal90_labels, skiprows=[10] )
pre_theta = pd.read_csv( f"{dir}/PSD/theta.csv", names = aal90_labels, skiprows=[10] )
dmt_alpha = pd.read_csv( f"{dir}/PSD/DMT_alpha.csv", names = aal90_labels, skiprows=[10] )
dmt_beta = pd.read_csv( f"{dir}/PSD/DMT_beta.csv", names = aal90_labels, skiprows=[10] )
dmt_delta = pd.read_csv( f"{dir}/PSD/DMT_delta.csv", names = aal90_labels, skiprows=[10] )
dmt_gamma1 = pd.read_csv( f"{dir}/PSD/DMT_gamma1.csv", names = aal90_labels, skiprows=[10] )
dmt_gamma2 = pd.read_csv( f"{dir}/PSD/DMT_gamma2.csv", names = aal90_labels, skiprows=[10] )
dmt_theta = pd.read_csv( f"{dir}/PSD/DMT_theta.csv", names = aal90_labels, skiprows=[10] )

In [70]:
print( dmt_alpha )

    Precentral_L  Precentral_R  Frontal_Sup_L  Frontal_Sup_R  \
0       0.412530      0.557590       0.261450       0.308940   
1       0.155180      0.156500       0.110360       0.109730   
2       0.120630      0.132540       0.094445       0.102430   
3       0.074214      0.095707       0.073736       0.080745   
4       0.197220      0.179980       0.118980       0.120590   
5       0.082214      0.110000       0.082678       0.093249   
6       0.297990      0.161910       0.204520       0.142460   
7       0.068437      0.072614       0.069650       0.073279   
8       0.080961      0.093203       0.072165       0.084189   
9       0.092613      0.093962       0.082399       0.081957   
10      0.114530      0.109920       0.084918       0.088906   
11      0.228210      0.273270       0.167780       0.195100   
12      0.394840      0.359850       0.262310       0.249100   
13      0.170010      0.189830       0.129820       0.148240   
14      0.155390      0.266180       0.1

In [ ]:
pre_alpha_m = pre_alpha.mean( axis=1 ).round( 3 )
pre_beta_m = pre_beta.mean( axis=1 ).round( 3 )
pre_delta_m = pre_delta.mean( axis=1 ).round( 3 )
pre_gamma1_m = pre_gamma1.mean( axis=1 ).round( 3 )
pre_gamma2_m = pre_gamma2.mean( axis=1 ).round( 3 )
pre_theta_m = pre_theta.mean( axis=1 ).round( 3 )
dmt_alpha_m = dmt_alpha.mean( axis=1 ).round( 3 )
dmt_beta_m = dmt_beta.mean( axis=1 ).round( 3 )
dmt_delta_m = dmt_delta.mean( axis=1 ).round( 3 )
dmt_gamma1_m = dmt_gamma1.mean( axis=1 ).round( 3 )
dmt_gamma2_m = dmt_gamma2.mean( axis=1 ).round( 3 )
dmt_theta_m = dmt_theta.mean( axis=1 ).round( 3 )

Functional connectivity: coherence

In [110]:
import mne
import scipy

eeg_data = {}

for i in range( 1, 36 ):
    if i not in [3, 6, 9, 14, 17, 24, 32]:
        if i in range( 1, 10 ):
            eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
        else:
            eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )


Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S01_DMT_ICA_pruned.set...
Not setting metadata
210 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S02_DMT_ICA_pruned.set...
Not setting metadata
172 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S04_DMT_ICA_pruned.set...
Not setting metadata
227 matching events found


/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: Data file name in EEG.data (S01-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S01_DMT_ICA_pruned.fdt).
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: Data file name in EEG.data (S02-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S02_DMT_IC

No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S05_DMT_ICA_pruned.set...
Not setting metadata
119 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S07_DMT_ICA_pruned.set...
Not setting metadata
134 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S08_DMT_ICA_pruned.set...
Not setting metadata
157 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S10_DMT_ICA_pruned.set...
Not setting metadata
104 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting pa

/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: Data file name in EEG.data (S05-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S05_DMT_ICA_pruned.fdt).
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S0{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:9: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S0{i}'] = mne.io.read_epochs_eeglab( f"

Not setting metadata
108 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S12_DMT_ICA_pruned.set...
Not setting metadata
149 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S13_DMT_ICA_pruned.set...
Not setting metadata
150 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S15_DMT_ICA_pruned.set...


...

/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: Data file name in EEG.data (S29-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S29_DMT_ICA_pruned.fdt).
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{di

Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S31_DMT_ICA_pruned.set...
Not setting metadata
160 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S33_DMT_ICA_pruned.set...
Not setting metadata
172 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S34_DMT_ICA_pruned.set...
Not setting metadata
168 matching events found
No baseline correction applied
0 projection items activated
Ready.


/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: Data file name in EEG.data (S31-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S31_DMT_ICA_pruned.fdt).
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: Data file name in EEG.data (S33-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S33_DMT_ICA_

Extracting parameters from /home/yijun-wu/Documents/24_25/research/ML_DMT/ML_DMT/CleanEEG/S35_DMT_ICA_pruned.set...
Not setting metadata
226 matching events found
No baseline correction applied
0 projection items activated
Ready.


/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )
/tmp/ipykernel_152482/2865371711.py:11: RuntimeWarning: Data file name in EEG.data (S35-DMT_ICA_pruned.fdt) is incorrect, the file name must have changed on disk, using the correct file name (S35_DMT_ICA_pruned.fdt).
  eeg_data[f'S{i}'] = mne.io.read_epochs_eeglab( f"{dir}/CleanEEG/S{i}_DMT_ICA_pruned.set" )


In [111]:
for data in eeg_data:
    print( np.shape( eeg_data[data] ) )

(210, 24, 1000)
(172, 24, 1000)
(227, 24, 1000)
(119, 24, 1000)
(134, 24, 1000)
(157, 24, 1000)
(104, 24, 1000)
(108, 24, 1000)
(149, 24, 1000)
(150, 24, 1000)
(272, 24, 1000)
(228, 24, 1000)
(254, 24, 1000)
(137, 24, 1000)
(192, 24, 1000)
(190, 24, 1000)
(226, 24, 1000)
(239, 24, 1000)
(125, 24, 1000)
(160, 24, 1000)
(164, 24, 1000)
(173, 24, 1000)
(155, 24, 1000)
(191, 24, 1000)
(160, 24, 1000)
(172, 24, 1000)
(168, 24, 1000)
(226, 24, 1000)


In [99]:
# Channeal coherence
from mne_connectivity import spectral_connectivity_epochs

def channel_coherence( data, f_min, f_max ):
    coh = spectral_connectivity_epochs( data, 
                                        method='imcoh',
                                        mode='multitaper',
                                        fmin=f_min, fmax=f_max,  
                                        faverage=True, 
                                        mt_adaptive=True )
    return coh

In [143]:
# regional coherence

def source_coherence( data, f_min, f_max, region1, region2 ):
    coh = spectral_connectivity_epochs( data, 
                                       method='imcoh',
                                       mode='multitaper',
                                       fmin=f_min, fmax=f_max,
                                       seed= region1,
                                       target= region2  
                                       )
    return coh

In [ ]:
s02 = eeg_data['S02'].pick(['all']).get_data()
s02_coh = spectral_connectivity_epochs( eeg_data['S02'], method='coh', fmin=1, fmax=50, faverage=True)


Connectivity computation...
only using indices for lower-triangular matrix
    computing connectivity for 276 connections
    using t=0.000s..1.998s for estimation (1000 points)
    frequencies: 1.0Hz..50.0Hz (99 points)
    connectivity scores will be averaged for each band
    Using multitaper spectrum estimation with 7 DPSS windows
    the following metrics will be computed: Coherence
    computing cross-spectral density for epoch 1


    computing cross-spectral density for epoch 2
    computing cross-spectral density for epoch 3
    computing cross-spectral density for epoch 4
    computing cross-spectral density for epoch 5
    computing cross-spectral density for epoch 6
    computing cross-spectral density for epoch 7
    computing cross-spectral density for epoch 8
    computing cross-spectral density for epoch 9
    computing cross-spectral density for epoch 10
    computing cross-spectral density for epoch 11
    computing cross-spectral density for epoch 12
    computing cross-spectral density for epoch 13
    computing cross-spectral density for epoch 14
    computing cross-spectral density for epoch 15
    computing cross-spectral density for epoch 16
    computing cross-spectral density for epoch 17
    computing cross-spectral density for epoch 18
    computing cross-spectral density for epoch 19
    computing cross-spectral density for epoch 20
    computing cross-spectral density for epoch 21


/tmp/ipykernel_152482/1408642267.py:2: RuntimeWarning: There were no Annotations stored in <EpochsEEGLAB | 172 events (all good), 0 – 1.998 s (baseline off), ~31.5 MiB, data loaded,
 'X/X': 172>, so metadata was not modified.
  s02_coh = spectral_connectivity_epochs( eeg_data['S02'], method='coh', indices=None, fmin=1, fmax=50, faverage=True)
/tmp/ipykernel_152482/1408642267.py:2: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  s02_coh = spectral_connectivity_epochs( eeg_data['S02'], method='coh', indices=None, fmin=1, fmax=50, faverage=True)


    computing cross-spectral density for epoch 22
    computing cross-spectral density for epoch 23
    computing cross-spectral density for epoch 24
    computing cross-spectral density for epoch 25
    computing cross-spectral density for epoch 26
    computing cross-spectral density for epoch 27
    computing cross-spectral density for epoch 28
    computing cross-spectral density for epoch 29
    computing cross-spectral density for epoch 30
    computing cross-spectral density for epoch 31
    computing cross-spectral density for epoch 32
    computing cross-spectral density for epoch 33
    computing cross-spectral density for epoch 34
    computing cross-spectral density for epoch 35
    computing cross-spectral density for epoch 36
    computing cross-spectral density for epoch 37
    computing cross-spectral density for epoch 38
    computing cross-spectral density for epoch 39
    computing cross-spectral density for epoch 40
    computing cross-spectral density for epoch 41


In [149]:
print( s02_coh.get_data( output='dense' ) )
# print( np.shape( s02_coh.get_data( output= 'dense' ) ))

[[[0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]]

 [[0.56588167]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]]

 [[0.7313753 ]
  [0.39936616]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0.        ]
  [0. 

In [142]:
from mne_connectivity.viz import plot_connectivity_circle

plot_connectivity_circle( s02_coh.get_data( output='dense' ), eeg_data['S02'].ch_names )
print( eeg_data['S02'].ch_names )

ValueError: con has to be 1D or a square matrix

In [71]:
regions = {
    'Fp1': 'Frontal_Sup_Medial_L',       # Left prefrontal cortex
    'Fp2': 'Frontal_Sup_Medial_R',       # Right prefrontal cortex
    'Fz': 'Frontal_Sup_Medial_L',        # Medial frontal cortex
    'F7': 'Frontal_Mid_L',               # Left lateral frontal cortex
    'F8': 'Frontal_Mid_R',               # Right lateral frontal cortex
    'FC1': 'Frontal_Sup_L',              # Left superior frontal cortex
    'FC2': 'Frontal_Sup_R',              # Right superior frontal cortex
    'Cz': 'Cingulum_Mid_L',              # Midline cingulate cortex
    'C3': 'Precentral_L',                # Left primary motor cortex
    'C4': 'Precentral_R',                # Right primary motor cortex
    'T7': 'Temporal_Sup_L',              # Left superior temporal gyrus
    'T8': 'Temporal_Sup_R',              # Right superior temporal gyrus
    'CPz': 'Cingulum_Post_L',            # Midline posterior cingulate cortex
    'CP1': 'Parietal_Sup_L',             # Left superior parietal lobule
    'CP2': 'Parietal_Sup_R',             # Right superior parietal lobule
    'CP5': 'SupraMarginal_L',            # Left supramarginal gyrus
    'CP6': 'SupraMarginal_R',            # Right supramarginal gyrus
    'TP9': 'Temporal_Inf_L',             # Left inferior temporal gyrus
    'TP10': 'Temporal_Inf_R',            # Right inferior temporal gyrus
    'Pz': 'Precuneus_L',                 # Medial parietal lobe (precuneus)
    'P3': 'Parietal_Inf_L',              # Left inferior parietal lobule
    'P4': 'Parietal_Inf_R',              # Right inferior parietal lobule
    'O1': 'Occipital_Inf_L',             # Left inferior occipital gyrus
    'O2': 'Occipital_Inf_R'              # Right inferior occipital gyrus
}

Nonlinear metrics: Spectral Entropy

In [49]:
def spectral_entropy( psd ):
    psd_norm = psd / np.sum( psd )
    psd_norm[psd_norm == 0] = np.finfo( float ).eps
    return -np.sum( psd_norm * np.log2( psd_norm ) )

In [61]:
# compute pre dmt entropy per band
entropy_pre_alpha = pre_alpha.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_pre_beta = pre_beta.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_pre_delta = pre_delta.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_pre_gamma1 = pre_gamma1.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_pre_gamma2 = pre_gamma2.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_pre_theta = pre_theta.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
# compute dmt entropy per band
entropy_dmt_alpha = dmt_alpha.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_dmt_beta = dmt_beta.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_dmt_delta = dmt_delta.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_dmt_gamma1 = dmt_gamma1.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_dmt_gamma2 = dmt_gamma2.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )
entropy_dmt_theta = dmt_theta.apply( spectral_entropy, axis = 1 ).to_numpy().reshape( -1, 1 )